# London School File Filtering
This notebook shows a simple Python workflow for filtering school performance files and keeping only records for London schools.

## 1. Load the London school list

## 2. Extract London school URNs

## 3. Process each file

## 4. Save filtered outputs

In [ ]:
import pandas as pd
from pathlib import Path


# =========================
# 1. Set file paths
# =========================
london_file = Path("data/London_school_info.xlsx")
input_folder = Path("data/performance_2025")
output_folder = Path("output/2025_London")
output_folder.mkdir(parents=True, exist_ok=True)


# =========================
# 2. Read the London school list and extract URNs
# =========================
london_df = pd.read_excel(london_file)

# Remove leading/trailing spaces from column names
london_df.columns = london_df.columns.str.strip()

# Check whether the URN column exists
if "URN" not in london_df.columns:
    raise ValueError("The file 'London_school_info.xlsx' does not contain a 'URN' column.")

# Extract London school URNs, remove missing values and duplicates
london_urns = set(
    pd.to_numeric(london_df["URN"], errors="coerce")
    .dropna()
    .astype(int)
)

print(f"Number of London school URNs: {len(london_urns)}")


# =========================
# 3. Define a function to process each file
# =========================
def process_file(file_path, output_folder, london_urns):
    try:
        # Read file depending on file type
        if file_path.suffix.lower() == ".csv":
            df = pd.read_csv(file_path, low_memory=False)
        elif file_path.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(file_path)
        else:
            print(f"Skipped unsupported file type: {file_path.name}")
            return

        # Remove leading/trailing spaces from column names
        df.columns = df.columns.str.strip()

        # Check whether the URN column exists
        if "URN" not in df.columns:
            print(f"File {file_path.name} does not contain a 'URN' column and was skipped.")
            return

        # Convert URN to numeric format for consistent filtering
        df["URN"] = pd.to_numeric(df["URN"], errors="coerce")

        # Keep only rows where the URN belongs to a London school
        df_london = df[df["URN"].isin(london_urns)].copy()

        # Convert URN back to integer type if the filtered file is not empty
        if not df_london.empty:
            df_london["URN"] = df_london["URN"].astype("Int64")

        # Define output file path
        output_file = output_folder / file_path.name

        # Save filtered file
        if file_path.suffix.lower() == ".csv":
            df_london.to_csv(output_file, index=False, encoding="utf-8-sig")
        else:
            df_london.to_excel(output_file, index=False)

        print(
            f"Processed: {file_path.name} | Original rows: {len(df)} | London rows: {len(df_london)}"
        )

    except Exception as e:
        print(f"Error processing file {file_path.name}: {e}")


# =========================
# 4. Loop through all files in the folder
# =========================
all_files = list(input_folder.iterdir())

print(f"Number of files found: {len(all_files)}")

for file in all_files:
    if file.is_file():
        process_file(file, output_folder, london_urns)

print("All files have been processed.")